<a href="https://colab.research.google.com/github/079bce159sarwagya-ctrl/Pytorch/blob/main/1.%20Foundation/NN_Module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import torch
device='cuda' if torch.cuda.is_available() else 'cpu'

df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [37]:
df.shape
#df.isnull().sum()

(569, 31)

In [11]:
df.drop(columns=['Unnamed: 32','id'], inplace=True)

Train Test split




In [12]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0], test_size=0.2, random_state=42)

Scaling

In [13]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

Label Encoder

In [14]:
encoder=LabelEncoder()
y_train=encoder.fit_transform(y_train)
y_test=encoder.transform(y_test)

Numpy arrays to Pytorch tensors

In [15]:
X_train_tensor=torch.tensor(X_train, dtype=torch.float32)
X_test_tensor=torch.tensor(X_test, dtype=torch.float32)
y_train_tensor=torch.tensor(y_train, dtype=torch.float32)
y_test_tensor=torch.tensor(y_test, dtype=torch.float32)

Neural Network Model

In [36]:
class Model(nn.Module):

    def __init__(self,num_para):

        super().__init__()

        self.network=nn.Sequential(
            nn.Linear(num_para, 10),
            nn.ReLU(),
            nn.Linear(10,5),
            nn.ReLU(),
            nn.Linear(5,1),
            nn.Sigmoid()
        )

    def forward(self, features):

            return self.network(features)


epochs= 25
learning_rate=0.1

X_train_device = X_train_tensor.to(device)
y_train_device = y_train_tensor.to(device).float()  # binary_cross_entropy expects float target


# Initialize the model
model = Model(X_train_device.shape[1]).to(device)
#print(f'Parameters: ', list(model.parameters()))
optimizer=optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(epochs):

    # Forward pass
    y_pred = model(X_train_device)

    # Loss calculation
    loss_fun=nn.BCELoss()
    loss = loss_fun(y_pred, y_train_device.unsqueeze(1))
    print(f"Epoch: {epoch}, loss: {loss.item()}")

    #Clear gradients
    optimizer.zero_grad()

    # Backward pass
    loss.backward()

    # Optimizers

    optimizer.step()


#print(f'final parameters: ', list(model.parameters()))

Epoch: 0, loss: 0.6658554673194885
Epoch: 1, loss: 0.5517865419387817
Epoch: 2, loss: 0.2965555489063263
Epoch: 3, loss: 0.1647026389837265
Epoch: 4, loss: 0.10722487419843674
Epoch: 5, loss: 0.08606727421283722
Epoch: 6, loss: 0.08433089405298233
Epoch: 7, loss: 0.08965002000331879
Epoch: 8, loss: 0.08454461395740509
Epoch: 9, loss: 0.07173434644937515
Epoch: 10, loss: 0.06350059062242508
Epoch: 11, loss: 0.058042462915182114
Epoch: 12, loss: 0.05046016350388527
Epoch: 13, loss: 0.04171333834528923
Epoch: 14, loss: 0.03414461389183998
Epoch: 15, loss: 0.03515484184026718
Epoch: 16, loss: 0.0285772904753685
Epoch: 17, loss: 0.030834631994366646
Epoch: 18, loss: 0.028323935344815254
Epoch: 19, loss: 0.023325525224208832
Epoch: 20, loss: 0.022354403510689735
Epoch: 21, loss: 0.021611563861370087
Epoch: 22, loss: 0.01833888702094555
Epoch: 23, loss: 0.018527565523982048
Epoch: 24, loss: 0.01702563464641571


Model evaluation


In [35]:
with torch.no_grad():
 y_pred=model(X_test_tensor.to(device))
 y_pred=0*(y_pred<0.5)+1*(y_pred>=0.5)
 evaluation_df = pd.DataFrame({
    'Y_test': y_test.flatten(),
    'Y_pred': y_pred.detach().cpu().numpy().flatten()
})

print( evaluation_df.head())

Accuracy= ( evaluation_df['Y_test'] == evaluation_df['Y_pred']).mean()
print(f"Accuracy: {Accuracy}")

   Y_test  Y_pred
0       0       0
1       1       1
2       1       1
3       0       0
4       0       0
Accuracy: 0.9736842105263158
